# Terminal Case Study — Phase 2: Multi-Lane Gate

**Case study**: Intermodal Container Terminal | **Phase**: 2 of 5

## Learning Objectives
By the end of this notebook you will be able to:
1. Model an M/M/c queue (multiple parallel gates) using `simdes`.
2. Compute the Erlang-C formula and compare it to simulation.
3. Demonstrate the super-linear benefit of adding a second gate.
4. Choose the minimum number of gates that satisfies a service-level target (Wq < 5 min).

---
> Phase 2 adds multiple gates (M/M/c).  The Erlang-C formula is exact for this system.

In [ ]:
import sys
from pathlib import Path
# Ensure course/ is on sys.path so the case_studies package is importable
_root = next(p for p in [Path.cwd()] + list(Path.cwd().parents) if (p / 'simdes').is_dir())
_course = _root / 'course'
if str(_course) not in sys.path:
    sys.path.insert(0, str(_course))


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from case_studies.terminal.terminal_model import run_terminal, TerminalParams
from simdes.analysis import confidence_interval

In [ ]:
def erlang_c(c: int, lam: float, mu: float) -> float:
    """Erlang-C formula — P(waiting) for an M/M/c queue.

    Args:
        c: Number of servers.
        lam: Arrival rate (same units as mu).
        mu: Per-server service rate.

    Returns:
        Probability that an arriving customer must wait (C(c, λ/μ)).
    """
    rho = lam / (c * mu)   # utilisation per server
    if rho >= 1.0:
        return 1.0
    a = lam / mu           # offered load (Erlangs)

    # Numerator: a^c / (c! * (1 - rho))
    numer = (a ** c) / (math.factorial(c) * (1 - rho))
    # Sum term: Σ_{k=0}^{c-1} a^k / k!
    denom = sum((a ** k) / math.factorial(k) for k in range(c)) + numer
    return numer / denom


def erlang_c_wq(c: int, lam: float, mu: float) -> float:
    """Mean waiting time in queue for M/M/c."""
    ec = erlang_c(c, lam, mu)
    return ec / (c * mu - lam)


# Baseline: λ = 10 trucks/hr, gate_mean = 5 min
lam = 10.0 / 60.0   # per minute
mu  = 1.0  / 5.0    # per minute

print('Gates | rho    | Erlang-C Wq (min)')
for c in range(1, 6):
    rho = lam / (c * mu)
    wq  = erlang_c_wq(c, lam, mu) if rho < 1 else float('inf')
    print(f'  {c}   | {rho:.3f} | {wq:.2f}')

In [ ]:
# Simulation sweep over number of gates
rows = []
for n_gates in range(1, 6):
    p = TerminalParams(n_gates=n_gates, n_cranes=100,
                       arrival_rate=10.0, gate_mean=5.0, crane_mean=0.1, sim_time=80.0)
    df_r = run_terminal(p, n_reps=20)
    m, lo, hi = confidence_interval(df_r['mean_wait_gate'].to_numpy())
    rows.append({'n_gates': n_gates, 'sim_Wq': m, 'ci_lo': lo, 'ci_hi': hi,
                 'theory_Wq': erlang_c_wq(n_gates, lam, mu)})

sweep = pd.DataFrame(rows)
sweep

In [ ]:
# Simulation vs. Erlang-C
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(sweep['n_gates'], sweep['theory_Wq'], 'k--s', label='Erlang-C theory', lw=1.5)
ax.errorbar(sweep['n_gates'], sweep['sim_Wq'],
            yerr=[sweep['sim_Wq']-sweep['ci_lo'], sweep['ci_hi']-sweep['sim_Wq']],
            fmt='o', color='tab:orange', capsize=4, label='Simulation (20 reps)')
ax.axhline(5, color='red', ls=':', lw=1.2, label='Target Wq = 5 min')
ax.set_xlabel('Number of gates')
ax.set_ylabel('Mean gate wait $W_q$ (min)')
ax.set_title('Phase 2 — Multi-lane gate: Erlang-C vs. simulation')
ax.legend()
ax.grid(alpha=0.2)
fig.tight_layout()
plt.show()

In [ ]:
# Marginal benefit of each additional gate
sweep['wq_reduction'] = sweep['sim_Wq'].diff().abs()
print('Marginal reduction in gate wait per additional gate:')
print(sweep[['n_gates', 'sim_Wq', 'wq_reduction']].to_string(index=False))

## Summary

Key findings:
- Simulation agrees closely with the Erlang-C formula — confirming the model is correct.
- The second gate provides a dramatic reduction in wait (super-linear benefit).
- To satisfy Wq < 5 min at λ = 10 trucks/hr, at least **3 gates** are needed.
- Marginal gains diminish after 3 gates: adding a 4th or 5th gate has limited impact.

In Phase 3 we add the crane/yard stage to complete the terminal model.

## Try It Yourself

1. What is the minimum number of gates to keep Wq < 2 minutes at λ = 15 trucks/hr?
2. Verify that the simulation still agrees with Erlang-C when arrival rate doubles.
3. The Erlang-C formula assumes exponential service times. What happens when
   service times follow a deterministic distribution (gate_cv = 0)?
   *(Hint: you cannot test this directly with TerminalParams, but can you argue theoretically?)*